# Computational Finance & Quantitative Modeling
## Master Notebook — End-to-End Walkthrough

This notebook connects every module in the project into a single end-to-end pipeline.

| Section | Topic |
|---|---|
| 1 | Real Market Data |
| 2 | Stochastic Processes |
| 3 | Option Pricing (Black-Scholes + Monte Carlo) |
| 4 | Volatility Models (GARCH + Heston) |
| 5 | Portfolio Optimization (Mean-Variance + Risk Parity) |
| 6 | Backtesting |
| 7 | Risk Report |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
import sys
from pathlib import Path

# ---- Path setup ----
# notebooks/ lives at project root, src/ and data/ and evaluation/ are siblings
ROOT = Path().resolve().parent
SRC  = ROOT / 'src'
for p in [str(ROOT), str(SRC),
          str(SRC / 'stochastic_processes'),
          str(SRC / 'option_pricing'),
          str(SRC / 'volatility_models'),
          str(SRC / 'portfolio_optimization')]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('Root:', ROOT)
print('Imports OK')

---
## 1. Real Market Data

We start by loading real price data for 6 ETFs covering the major asset classes.
All downstream modules use these returns and covariance estimates.

In [ ]:
from data.market_data import get_market_data, plot_prices, plot_correlation_matrix, plot_rolling_volatility

data = get_market_data()
prices    = data['prices']
returns   = data['returns']
mu        = data['mu']
cov       = data['cov']
names     = data['names']
tickers   = data['tickers']

print(f'\nDate range: {returns.index[0].date()} → {returns.index[-1].date()}')
print(f'Assets: {names}')
print(f'Observations: {len(returns):,} trading days')

In [ ]:
# Normalized price history
plot_prices(prices, names)

In [ ]:
# Correlation matrix
plot_correlation_matrix(returns, names)

In [ ]:
# Rolling volatility — shows volatility clustering in real data
plot_rolling_volatility(returns, names)

---
## 2. Stochastic Processes

Before pricing anything, we need to understand the models that generate price paths.
Brownian motion is the raw noise engine. GBM turns that noise into realistic price dynamics.

In [ ]:
from brownian_motion import simulate_brownian_motion
from geometric_bm import simulate_gbm

# ---- Brownian Motion ----
t_bm, W = simulate_brownian_motion(T=1.0, N=252, n_paths=20, seed=42)

plt.figure(figsize=(10, 5))
for i in range(W.shape[0]):
    plt.plot(t_bm, W[i], alpha=0.6, linewidth=0.8)
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.title('Brownian Motion — Pure Randomness')
plt.xlabel('Time')
plt.ylabel('W(t)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---- GBM: Asset Price Paths ----
t_gbm, S = simulate_gbm(S0=100, mu=0.08, sigma=0.2, T=1.0, N=252, n_paths=30, seed=42)

plt.figure(figsize=(10, 5))
for i in range(S.shape[0]):
    plt.plot(t_gbm, S[i], alpha=0.5, linewidth=0.8)
plt.plot(t_gbm, S.mean(axis=0), color='black', linewidth=2,
         linestyle='--', label='Mean path')
plt.title('Geometric Brownian Motion — Asset Price Paths')
plt.xlabel('Time (years)')
plt.ylabel('Price S(t)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Verify: Log returns are normally distributed ----
_, S_large = simulate_gbm(S0=100, mu=0.08, sigma=0.2, T=1.0, N=252, n_paths=20_000, seed=7)
log_ret = np.log(S_large[:, -1] / 100)
mean_theory = (0.08 - 0.5 * 0.2**2) * 1.0
std_theory  = 0.2 * np.sqrt(1.0)
x = np.linspace(log_ret.min(), log_ret.max(), 300)

plt.figure(figsize=(9, 5))
plt.hist(log_ret, bins=80, density=True, alpha=0.6, color='steelblue', label='Simulated')
plt.plot(x, norm.pdf(x, mean_theory, std_theory), color='crimson',
         linewidth=2, label='Theoretical N(μ, σ²)')
plt.xlabel('log(S(T)/S₀)')
plt.ylabel('Density')
plt.title('GBM: Log Returns are Normally Distributed')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Option Pricing — Black-Scholes & Monte Carlo

With GBM as our price model, we can price European options analytically (Black-Scholes)
and numerically (Monte Carlo). Both should give the same answer.

In [ ]:
from black_scholes import call_price, put_price, delta_call, gamma, vega, theta_call, put_call_parity_check
from monte_carlo_pricing import mc_call_price

S0, K, r, sigma, T = 100.0, 100.0, 0.05, 0.2, 1.0

bs_call = call_price(S0, K, T, r, sigma)
bs_put  = put_price(S0, K, T, r, sigma)
mc_call, mc_se = mc_call_price(S0, K, r, sigma, T, n_paths=100_000)

print(f'=== Option Pricing (S={S0}, K={K}, r={r}, σ={sigma}, T={T}) ===')
print(f'Black-Scholes Call:  {bs_call:.4f}')
print(f'Monte Carlo Call:    {mc_call:.4f}  (±{1.96*mc_se:.4f} 95% CI)')
print(f'Black-Scholes Put:   {bs_put:.4f}')
print()
put_call_parity_check(S0, K, T, r, sigma)

In [ ]:
# ---- Greeks across spot prices ----
S_vals = np.linspace(60, 140, 300)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, fn, label, color in zip(
    axes,
    [delta_call, gamma, vega, theta_call],
    ['Delta', 'Gamma', 'Vega', 'Theta'],
    ['steelblue', 'darkorange', 'green', 'purple']
):
    ax.plot(S_vals, fn(S_vals, K, T, r, sigma), color=color, linewidth=2)
    ax.axvline(K, color='gray', linestyle=':', linewidth=1)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(label)
    ax.set_xlabel('Stock Price S')
    ax.grid(alpha=0.3)

plt.suptitle(f'Black-Scholes Greeks  (K={K}, T={T}, r={r}, σ={sigma})', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Call price surface ----
import plotly.graph_objects as go

S_grid, sigma_grid = np.meshgrid(np.linspace(50, 150, 60), np.linspace(0.05, 0.6, 60))
V = call_price(S_grid, K, T, r, sigma_grid)

fig = go.Figure(data=[go.Surface(x=S_grid, y=sigma_grid, z=V, colorscale='Viridis')])
fig.update_layout(
    title='Black-Scholes Call Price Surface',
    scene=dict(xaxis_title='S', yaxis_title='σ', zaxis_title='Call Price')
)
fig.show()

---
## 4. Volatility Models — GARCH & Heston

Black-Scholes assumes constant volatility. In reality volatility clusters and is itself random.
GARCH models it as discrete time-varying. Heston models it as a continuous stochastic process.

In [ ]:
from garch import simulate_garch
from stochastic_volatility import simulate_heston

# ---- GARCH(1,1) ----
returns_garch, vol_garch, _ = simulate_garch(
    n=1000, omega=0.0001, alpha=0.10, beta=0.85, seed=42
)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(returns_garch, color='steelblue', linewidth=0.7, alpha=0.85)
axes[0].set_title('GARCH(1,1): Simulated Returns')
axes[0].set_ylabel('Return')
axes[0].grid(alpha=0.3)
axes[1].plot(vol_garch, color='crimson', linewidth=0.9)
axes[1].set_title('GARCH(1,1): Conditional Volatility — Clustering Effect')
axes[1].set_ylabel('σ_t')
axes[1].set_xlabel('Time Step')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Heston Stochastic Volatility ----
t_h, S_h, v_h = simulate_heston(
    S0=100, mu=0.05, v0=0.04,
    kappa=2.0, theta=0.04, xi=0.3, rho=-0.7,
    T=1.0, N=252, n_paths=1, seed=42
)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(t_h, S_h[0], color='steelblue', linewidth=1.2)
axes[0].set_title('Heston Model: Asset Price Path')
axes[0].set_ylabel('Price S(t)')
axes[0].grid(alpha=0.3)
axes[1].plot(t_h, np.sqrt(v_h[0]), color='crimson', linewidth=1.0)
axes[1].set_title('Heston Model: Stochastic Volatility √v(t)')
axes[1].set_ylabel('Volatility')
axes[1].set_xlabel('Time (years)')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Compare terminal distributions: BS vs GARCH vs Heston ----
n_paths = 20_000
S0_c, mu_c, sigma_c, T_c, N_c = 100.0, 0.05, 0.2, 1.0, 252
dt = T_c / N_c
v0 = sigma_c ** 2

# Black-Scholes
rng = np.random.default_rng(42)
Z = rng.standard_normal((n_paths, N_c))
S_BS = S0_c * np.exp(((mu_c - 0.5*sigma_c**2)*dt + sigma_c*np.sqrt(dt)*Z).sum(axis=1))

# Heston
_, S_heston, _ = simulate_heston(
    S0=S0_c, mu=mu_c, v0=v0,
    kappa=2.0, theta=v0, xi=0.3, rho=-0.7,
    T=T_c, N=N_c, n_paths=n_paths, seed=42
)
S_H = S_heston[:, -1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, S_T, label, color in zip(
    axes,
    [S_BS, S_H],
    ['Black-Scholes (constant σ)', 'Heston (stochastic σ)'],
    ['steelblue', 'crimson']
):
    ax.hist(S_T, bins=80, density=True, color=color, alpha=0.7)
    ax.axvline(S_T.mean(), color='black', linestyle='--', linewidth=1.2,
               label=f'Mean={S_T.mean():.1f}')
    ax.set_title(label)
    ax.set_xlabel('Terminal Price S(T)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Density')
plt.suptitle('Terminal Price Distributions: BS vs Heston', fontsize=12)
plt.tight_layout()
plt.show()

---
## 5. Portfolio Optimization

Using real market data, we build and compare four portfolio strategies:
equal weight, minimum variance, maximum Sharpe, and risk parity.

In [ ]:
from mean_variance import (
    minimum_variance_portfolio, maximum_sharpe_portfolio,
    compute_efficient_frontier, simulate_random_portfolios,
    portfolio_volatility, portfolio_return, portfolio_sharpe
)
from risk_parity import equal_risk_contribution_portfolio, naive_risk_parity, risk_contribution_pct

n = len(names)
rf = 0.02

mvp = minimum_variance_portfolio(mu, cov)
msr = maximum_sharpe_portfolio(mu, cov, rf=rf)
erc = equal_risk_contribution_portfolio(cov)
w_ew = np.ones(n) / n

print('=== Minimum Variance Portfolio ===')
for name, w in zip(names, mvp['weights']):
    print(f'  {name:<15} {w:.1%}')
print(f'  Return: {mvp["return"]:.2%}  Vol: {mvp["volatility"]:.2%}  Sharpe: {mvp["sharpe"]:.2f}')
print()
print('=== Maximum Sharpe Portfolio ===')
for name, w in zip(names, msr['weights']):
    print(f'  {name:<15} {w:.1%}')
print(f'  Return: {msr["return"]:.2%}  Vol: {msr["volatility"]:.2%}  Sharpe: {msr["sharpe"]:.2f}')
print()
print('=== Equal Risk Contribution Portfolio ===')
for name, w, rc in zip(names, erc['weights'], erc['risk_contributions']):
    print(f'  {name:<15} weight={w:.1%}  risk contrib={rc:.1%}')

In [ ]:
# ---- Efficient Frontier ----
sim = simulate_random_portfolios(mu, cov, n_portfolios=5000, rf=rf)
frontier_vols, frontier_rets = compute_efficient_frontier(mu, cov)

fig, ax = plt.subplots(figsize=(11, 7))
ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#1a1a2e')

sc = ax.scatter(sim['vols'], sim['returns'], c=sim['sharpes'],
                cmap='viridis', alpha=0.3, s=8)
plt.colorbar(sc, ax=ax, label='Sharpe Ratio')
ax.plot(frontier_vols, frontier_rets, color='white', linewidth=2.5, label='Efficient Frontier')
ax.scatter(mvp['volatility'], mvp['return'], color='cyan', s=150, zorder=10,
           label=f'Min Variance  Sharpe={mvp["sharpe"]:.2f}')
ax.scatter(msr['volatility'], msr['return'], color='red', s=200, marker='*', zorder=10,
           label=f'Max Sharpe  Sharpe={msr["sharpe"]:.2f}')

asset_vols = np.sqrt(np.diag(cov))
for i, name in enumerate(names):
    ax.scatter(asset_vols[i], mu[i], s=80, zorder=8, marker='D')
    ax.annotate(name, (asset_vols[i], mu[i]),
                textcoords='offset points', xytext=(6,4), fontsize=9, color='white')

for spine in ax.spines.values(): spine.set_edgecolor('gray')
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white')
ax.title.set_color('white')
ax.set_xlabel('Portfolio Volatility (σ)')
ax.set_ylabel('Expected Return (μ)')
ax.set_title('Efficient Frontier — Real Market Data')
ax.legend(fontsize=9, facecolor='#2a2a3e', labelcolor='white')
ax.grid(alpha=0.2, color='gray')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Risk contribution comparison ----
strategies_w = {'Equal Weight': w_ew, 'Min Variance': mvp['weights'],
                'Max Sharpe': msr['weights'], 'ERC': erc['weights']}
colors_bar = ['steelblue', 'darkorange', 'crimson', 'green']
x = np.arange(n)
width = 0.2

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for i, (label, w) in enumerate(strategies_w.items()):
    axes[0].bar(x + i*width, w, width, label=label, color=colors_bar[i], alpha=0.8)
    axes[1].bar(x + i*width, risk_contribution_pct(w, cov), width,
                label=label, color=colors_bar[i], alpha=0.8)

for ax, title in zip(axes, ['Portfolio Weights', 'Risk Contributions (%)']):
    ax.set_xticks(x + width*1.5)
    ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, axis='y')

axes[1].axhline(1/n, color='black', linestyle='--', linewidth=1, label=f'Equal={1/n:.1%}')
plt.suptitle('Weights vs Risk Contributions by Strategy', fontsize=12)
plt.tight_layout()
plt.show()

---
## 6. Backtesting

We test each strategy on real historical data using a rolling-window backtest.
At each monthly rebalance, we re-estimate parameters using the past year of data
and hold the new weights until the next rebalance.

In [ ]:
from evaluation.backtesting import run_all_strategies

print('Running backtests (lookback=252 days, monthly rebalance)...\n')
backtest_results = run_all_strategies(
    returns,
    lookback=252,
    rebalance_freq=21,
    rf_annual=0.02,
    transaction_cost=0.001,
)

In [ ]:
# ---- Equity curves ----
plt.figure(figsize=(12, 6))
for label, r in backtest_results.items():
    cum = (1 + r).cumprod()
    plt.plot(cum.index, cum, linewidth=1.5, label=label)
plt.xlabel('Date')
plt.ylabel('Portfolio Value ($1 invested)')
plt.title('Backtest Equity Curves — Real Market Data')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Drawdowns ----
from evaluation.risk_metrics import drawdown_series

plt.figure(figsize=(12, 5))
for label, r in backtest_results.items():
    dd = drawdown_series(r)
    plt.plot(dd.index, dd * 100, linewidth=1.2, label=label)
plt.axhline(0, color='black', linewidth=0.5)
plt.xlabel('Date')
plt.ylabel('Drawdown (%)')
plt.title('Strategy Drawdowns')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Final Risk Report

Full risk metrics for every strategy — return, risk, ratios, tail risk.

In [ ]:
from evaluation.risk_metrics import risk_report

reports = {name: risk_report(r, name=name) for name, r in backtest_results.items()}
summary = pd.DataFrame(reports).T
pd.set_option('display.float_format', '{:.4f}'.format)
print('=== Final Risk Report ===\n')
print(summary.to_string())

In [ ]:
# ---- Visual summary ----
metrics = [
    ('Ann. Return',     'steelblue',  True),
    ('Ann. Volatility', 'crimson',    True),
    ('Sharpe Ratio',    'green',      False),
    ('Sortino Ratio',   'darkorange', False),
    ('Calmar Ratio',    'purple',     False),
    ('Max Drawdown',    'red',        True),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for ax, (metric, color, as_pct) in zip(axes, metrics):
    vals = summary[metric].astype(float)
    ax.bar(range(len(summary)), vals, color=color, alpha=0.8)
    ax.set_xticks(range(len(summary)))
    ax.set_xticklabels(summary.index, rotation=20, ha='right', fontsize=8)
    ax.set_title(metric, fontsize=10)
    ax.grid(alpha=0.3, axis='y')
    if as_pct:
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1%}'))

plt.suptitle('Final Risk Report — Strategy Comparison', fontsize=13)
plt.tight_layout()
plt.show()

---
## Summary

| Module | What was built | Key output |
|---|---|---|
| **Stochastic Processes** | BM, GBM, Monte Carlo | Price path simulation |
| **Option Pricing** | Black-Scholes, Greeks, MC pricing | Closed-form + numerical option prices |
| **Volatility Models** | GARCH, Heston | Time-varying and stochastic vol |
| **Portfolio Optimization** | Mean-Variance, Risk Parity | Efficient frontier, ERC weights |
| **Backtesting** | Rolling-window strategy backtest | Equity curves, drawdowns |
| **Risk Metrics** | Sharpe, Sortino, Calmar, VaR, CVaR | Full risk report per strategy |

The project builds from first principles — pure randomness (Brownian motion) → asset prices (GBM) → option pricing (Black-Scholes) → realistic volatility (GARCH/Heston) → portfolio construction (Markowitz/ERC) → historical evaluation (backtesting + risk metrics).